In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

import matplotlib.pyplot as plt
import pandas as pd
import re
import os

In [3]:
spark = SparkSession.builder.getOrCreate()

In [4]:
parquet_path = "/content/drive/MyDrive/data_parquet/software_cleaned.parquet"
input_path = "/content/drive/MyDrive/data - MoMD/Software.jsonl"

In [16]:
df_data = spark.read.format("json").option("schema","True").load(input_path)

In [ ]:
df_data.show(truncate=False)

+----------+------------+------+-----------+------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------

In [7]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s\.\!\?]', '', text)
    return text.strip()
clean_udf = udf(clean_text, StringType())

In [8]:
def split_sentences(text):
    if text is None:
        return []
    sentences = re.split(r'[.!?]+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 0]
split_udf = udf(split_sentences)

In [17]:
df_progress_data = df_data.withColumn("review_text", concat_ws(" ", col("title"),col("text"))) \
                          .filter(trim(col("review_text")) != "") \
                          .dropDuplicates(["parent_asin", "review_text"])

In [18]:
df_split_data = df_progress_data.withColumn("review_text", clean_udf(col("review_text"))) \
                                .withColumn("sentences_text", split_udf(col("review_text"))) \
                                .withColumn("sentence_id", monotonically_increasing_id()+1) \
                                .select(
                                    col("parent_asin"),
                                    col("asin").alias("review_id"),
                                    col("sentence_id"),
                                    col("sentences_text"),
                                    col("rating")
                                )

In [ ]:
df_progress_data.show(5)

+-----------+----------+-----------+--------------------+------+
|parent_asin| review_id|sentence_id|      sentences_text|rating|
+-----------+----------+-----------+--------------------+------+
| 0005162092|0005162092|          1|[five stars it wo...|   5.0|
| 0071480935|0071480935|          2|[this is one of m...|   5.0|
| 0131817949|0131817949|          3|[basic but boring...|   3.0|
| 0133727866|0133727866|          4|[on target assist...|   5.0|
| 013446950X|013446950X|          5|[dont get it the ...|   1.0|
+-----------+----------+-----------+--------------------+------+
only showing top 5 rows


In [ ]:
df_progress_data.write.mode("overwrite").parquet("/content/drive/MyDrive/data_parquet/software_cleaned.parquet")

In [11]:
def get_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total / (1024**3)

In [19]:
initial_count = df_data.count()
after_clean_count = df_progress_data.count()
sentence_count = df_split_data.count()

In [20]:
input_size = os.path.getsize(input_path) / (1024**3)
output_size = get_size(parquet_path)

In [21]:
report = f"""
Số review ban đầu: {initial_count:,}
Số review còn lại: {after_clean_count:,}
Số câu sau khi tách: {sentence_count:,}
Kích thước ban đầu: {input_size:.2f}GB
Kích thước cuối cùng: {output_size:.2f}GB
"""
print(report)


Số review ban đầu: 4,880,181
Số review còn lại: 4,730,672
Số câu sau khi tách: 4,730,672
Kích thước ban đầu: 1.74GB
Kích thước cuối cùng: 0.43GB



In [23]:
report_path = "reviews_cleaning1_report_software.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)